# IoTSpy — Sample Data Generator

Generates a **synthetic but realistic** Parquet dataset for use with notebooks 01–03.

No real device data is used. Hostnames, IPs, and payload patterns are all invented to
approximate the distributions seen in typical IoT consumer device traffic.

**Run this notebook first**, then open notebooks 01–03.

Output: `../data/sample_captures.parquet`

---
### Synthetic device profiles

| Device | IP | Type | Notes |
|---|---|---|---|
| Smart TV | 192.168.1.10 | Consumer electronics | Ad-supported streaming + usage telemetry |
| Voice assistant | 192.168.1.20 | Smart speaker | Always-on; frequent cloud sync + ad targeting |
| Security camera | 192.168.1.30 | IP camera | Periodic health checks + video chunk uploads |
| Game console | 192.168.1.40 | Gaming | Heavy ad-SDK traffic during free-to-play sessions |

In [1]:
from pathlib import Path
import random, math, json
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd

# Reproducible
RNG = np.random.default_rng(seed=42)
random.seed(42)

OUT_PATH = Path("../data/sample_captures.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Session starts at a fixed anchor so the notebook is deterministic
SESSION_START = datetime(2024, 3, 15, 9, 0, 0, tzinfo=timezone.utc)
SESSION_DURATION_H = 2

print("Generating synthetic IoT traffic dataset...")

Generating synthetic IoT traffic dataset...


## 1  Define synthetic host catalogue

In [2]:
# Each entry: (hostname, category, is_known_broker, typical_method, port, is_tls)
HOST_CATALOGUE = [
    # ── Device vendor APIs ───────────────────────────────────────────────────
    ("api.acme-tv.example.com",         "VendorAPI",   False, "POST", 443, True),
    ("telemetry.acme-tv.example.com",   "VendorAPI",   False, "POST", 443, True),
    ("cdn.acme-tv.example.com",         "VendorCDN",   False, "GET",  443, True),
    ("api.vocalize-home.example.com",   "VendorAPI",   False, "POST", 443, True),
    ("sync.vocalize-home.example.com",  "VendorAPI",   False, "POST", 443, True),
    ("api.watchguard-cam.example.com",  "VendorAPI",   False, "POST", 443, True),
    ("clips.watchguard-cam.example.com","VendorCDN",   False, "PUT",  443, True),
    ("api.playvault.example.com",       "VendorAPI",   False, "POST", 443, True),
    ("cdn.playvault.example.com",       "VendorCDN",   False, "GET",  443, True),

    # ── Ad / tracking networks (known brokers) ────────────────────────────────
    ("ads.quantumreach.example.net",    "AdNetwork",   True,  "GET",  443, True),
    ("events.quantumreach.example.net", "AdNetwork",   True,  "POST", 443, True),
    ("track.pixelwave.example.io",      "AdNetwork",   True,  "POST", 443, True),
    ("bid.rtbfusion.example.com",       "AdNetwork",   True,  "POST", 443, True),
    ("collect.insightsdash.example.com","Analytics",   True,  "POST", 443, True),
    ("log.behavioriq.example.com",      "Analytics",   True,  "POST", 443, True),
    ("p.attribchain.example.com",       "Attribution", True,  "POST", 443, True),
    ("sdk.attribchain.example.com",     "Attribution", True,  "GET",  443, True),

    # ── Infrastructure / platform (benign) ────────────────────────────────────
    ("time.internal.example.net",       "NTP",         False, "GET",  80,  False),
    ("updates.example-firmware.com",    "OTA",         False, "GET",  443, True),
    ("ocsp.example-ca.com",             "OCSP",        False, "GET",  80,  False),
    ("connectivity.example-check.com",  "ConnCheck",   False, "GET",  443, True),

    # ── Weak-TLS / plaintext endpoints (risk signals) ─────────────────────────
    ("legacy-api.acme-tv.example.com",  "VendorAPI",   False, "POST", 80,  False),  # HTTP!
    ("reporting.datalink.example.net",  "Analytics",   True,  "POST", 80,  False),  # HTTP tracker
]

## 2  Define device traffic profiles

In [3]:
DEVICES = [
    {"ip": "192.168.1.10", "name": "smart-tv",        "vendor": "AcmeTV",      "security_score": 62},
    {"ip": "192.168.1.20", "name": "voice-assistant",  "vendor": "VocalizeHome","security_score": 55},
    {"ip": "192.168.1.30", "name": "security-camera",  "vendor": "WatchGuardCam","security_score": 71},
    {"ip": "192.168.1.40", "name": "game-console",     "vendor": "PlayVault",   "security_score": 48},
]

# Host affinity: each device has a weighted distribution over the host catalogue
# Indices map to HOST_CATALOGUE above
DEVICE_HOST_WEIGHTS = {
    "smart-tv":       [8,5,4, 0,0, 0,0, 0,0,  6,5,3,4, 4,3, 2,2,  2,1,1,1, 2,2],
    "voice-assistant":[0,0,0, 8,5, 0,0, 0,0,  4,5,4,3, 5,4, 3,2,  2,1,1,2, 0,1],
    "security-camera":[0,0,0, 0,0, 7,5, 0,0,  1,1,1,1, 2,1, 1,1,  3,3,2,2, 2,1],
    "game-console":   [0,0,0, 0,0, 0,0, 8,6,  8,7,6,7, 3,5, 4,3,  1,2,1,1, 1,1],
}

# Approximate requests per device over the session
DEVICE_VOLUME = {
    "smart-tv":       650,
    "voice-assistant": 480,
    "security-camera": 320,
    "game-console":    750,
}

## 3  Generate captures

In [4]:
import uuid

TLS_CIPHERS = [
    "TLS_AES_256_GCM_SHA384",
    "TLS_AES_128_GCM_SHA256",
    "TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384",
    "TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256",
]
TLS_VERSIONS = ["Tls13", "Tls13", "Tls12", "Tls12"]

AD_PATHS = [
    "/v1/events", "/v2/bid", "/collect", "/track", "/impression",
    "/sdk/init", "/sdk/log", "/pixel", "/beacon", "/v3/attribution",
]
VENDOR_PATHS = [
    "/api/v1/state", "/api/v2/sync", "/telemetry/batch", "/health",
    "/session/start", "/session/end", "/metrics", "/events",
    "/clip/upload", "/config", "/ping",
]
CDN_PATHS = [
    "/assets/thumb_{}.jpg", "/content/clip_{}.mp4", "/firmware/v{}.bin",
    "/images/splash.png", "/bundles/sdk-{}.js",
]


def sample_path(category: str) -> str:
    if category in ("AdNetwork", "Analytics", "Attribution"):
        return random.choice(AD_PATHS)
    if category in ("VendorCDN", "OTA"):
        p = random.choice(CDN_PATHS)
        return p.format(random.randint(1000, 9999))
    return random.choice(VENDOR_PATHS)


def sample_sizes(category: str, method: str):
    """Returns (request_body_bytes, response_body_bytes)."""
    if category == "VendorCDN":
        return 0, int(RNG.lognormal(13, 1.5))   # large response (assets/video)
    if method == "POST" and category in ("AdNetwork", "Analytics", "Attribution"):
        req = int(RNG.lognormal(7, 1.2))         # medium POST body (SDK payload)
        return req, int(RNG.lognormal(4, 0.8))
    if method == "POST":
        req = int(RNG.lognormal(5, 1.0))
        return req, int(RNG.lognormal(5, 1.0))
    return 0, int(RNG.lognormal(6, 1.2))


def sample_headers(category: str, method: str) -> tuple[dict, dict]:
    req = {"Accept": "application/json"}
    if method == "POST":
        req["Content-Type"] = "application/json"
    if category in ("AdNetwork", "Analytics", "Attribution"):
        # SDKs typically send device-ID headers
        req["X-Device-Id"] = str(uuid.uuid4())
        req["User-Agent"] = "IoTDeviceSDK/3.2.1"
    resp = {"Content-Type": "application/json" if category != "VendorCDN" else "video/mp4"}
    return req, resp


def inject_pii_risk(category: str, path: str, req_body: int) -> tuple[list, float]:
    """Return (tags, risk_score) using rule-like heuristics."""
    tags, score = [], 0.0
    if category in ("AdNetwork", "Analytics", "Attribution"):
        tags.append("DataBroker")
        score += 0.4
    # Large POST to ad-network = exfiltration signal
    if req_body > 8000 and category in ("AdNetwork", "Analytics", "Attribution"):
        tags.append("ExfiltrationRisk")
        score += 0.3
    # Device-ID header paths → PII risk
    if category in ("AdNetwork", "Attribution") and "/sdk/" in path:
        if RNG.random() < 0.4:
            tags.append("PiiDetected")
            score += 0.15
    return list(set(tags)), min(round(score, 4), 1.0)


rows = []
session_ms = int(SESSION_START.timestamp() * 1000)

for device in DEVICES:
    name = device["name"]
    n_requests = DEVICE_VOLUME[name]
    weights = DEVICE_HOST_WEIGHTS[name]
    total_w = sum(weights)
    probs = [w / total_w for w in weights]

    # Timestamps: Poisson arrivals with occasional bursts
    duration_ms = SESSION_DURATION_H * 3600 * 1000
    ts_offsets = sorted(RNG.integers(0, duration_ms, size=n_requests).tolist())

    for ts_offset in ts_offsets:
        host_idx = RNG.choice(len(HOST_CATALOGUE), p=probs)
        host, category, is_broker, default_method, port, is_tls = HOST_CATALOGUE[host_idx]

        method = default_method
        path   = sample_path(category)
        req_sz, resp_sz = sample_sizes(category, method)
        req_h, resp_h   = sample_headers(category, method)
        duration_ms_val = max(10, int(RNG.lognormal(4.5, 0.9)))

        cipher = random.choice(TLS_CIPHERS) if is_tls else ""
        tls_ver = TLS_VERSIONS[TLS_CIPHERS.index(cipher)] if cipher else ""

        status = 200
        if RNG.random() < 0.03:
            status = random.choice([400, 404, 500, 503])
        elif RNG.random() < 0.04:
            status = 204

        tags, risk_score = inject_pii_risk(category, path, req_sz)

        rows.append({
            "capture_id":          str(uuid.uuid4()),
            "host":                host,
            "port":                port,
            "method":              method,
            "scheme":              "https" if is_tls else "http",
            "path":                path,
            "status_code":         status,
            "protocol":            "HTTP",
            "is_tls":              int(is_tls),
            "tls_version":         tls_ver,
            "tls_cipher_suite":    cipher,
            "request_body_size":   req_sz,
            "response_body_size":  resp_sz,
            "duration_ms":         duration_ms_val,
            "client_ip":           device["ip"],
            "device_name":         name,
            "device_vendor":       device["vendor"],
            "device_security_score": device["security_score"],
            "is_modified":         0,
            "timestamp_ms":        session_ms + ts_offset,
            "category":            category,
            "is_known_broker":     int(is_broker),
            "tags":                json.dumps(tags),
            "risk_score":          risk_score,
            "request_headers":     json.dumps(req_h),
            "response_headers":    json.dumps(resp_h),
        })

df = pd.DataFrame(rows)
df["timestamp"] = pd.to_datetime(df["timestamp_ms"], unit="ms", utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Generated {len(df):,} synthetic captures across {df['client_ip'].nunique()} devices")
print(f"Unique hosts: {df['host'].nunique()}  |  "
      f"Session: {df['timestamp'].min().strftime('%H:%M')}–{df['timestamp'].max().strftime('%H:%M')} UTC")

Generated 2,200 synthetic captures across 4 devices
Unique hosts: 23  |  Session: 09:00–10:59 UTC


## 4  Preview and save

In [5]:
import json as _j
from collections import Counter
from itertools import chain

print("Category distribution:")
print(df["category"].value_counts().to_string())
print()
print("Tag distribution:")
all_tags = Counter(chain.from_iterable(df["tags"].apply(_j.loads)))
for tag, cnt in all_tags.most_common():
    print(f"  {tag:<22} {cnt:>5,}  ({cnt/len(df)*100:.1f}%)")
print()
print(f"Known-broker traffic: {df['is_known_broker'].sum():,}  ({df['is_known_broker'].mean()*100:.1f}%)")
print(f"Plaintext HTTP:       {(df['is_tls']==0).sum():,}  ({(df['is_tls']==0).mean()*100:.1f}%)")

Category distribution:
category
AdNetwork      769
VendorAPI      489
Analytics      352
VendorCDN      180
Attribution    177
NTP             79
OTA             64
ConnCheck       47
OCSP            43

Tag distribution:
  DataBroker             1,298  (59.0%)
  PiiDetected               93  (4.2%)
  ExfiltrationRisk          46  (2.1%)

Known-broker traffic: 1,298  (59.0%)
Plaintext HTTP:       225  (10.2%)


In [6]:
df.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(df):,} rows → {OUT_PATH.resolve()}")
print(f"Columns: {df.columns.tolist()}")

Saved 2,200 rows → /Users/annalise/git/aarnold-livefront/iotspy/analytics/notebooks/data/sample_captures.parquet
Columns: ['capture_id', 'host', 'port', 'method', 'scheme', 'path', 'status_code', 'protocol', 'is_tls', 'tls_version', 'tls_cipher_suite', 'request_body_size', 'response_body_size', 'duration_ms', 'client_ip', 'device_name', 'device_vendor', 'device_security_score', 'is_modified', 'timestamp_ms', 'category', 'is_known_broker', 'tags', 'risk_score', 'request_headers', 'response_headers', 'timestamp']
